In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import hashlib
import warnings
warnings.filterwarnings('ignore')

OCT_PATH  = "../data/OCT_aug"
XRAY_PATH = "../data/xray_aug"
SPLITS    = ['train', 'val', 'test']
OUTPUT_DIR = "../powerbi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Dossier PowerBI prêt :", OUTPUT_DIR)

✅ Dossier PowerBI prêt : ../powerbi


 CSV 1 : Distribution des classes

In [2]:
def export_distribution(base_path, dataset_name):
    records = []
    for split in SPLITS:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            continue
        classes = sorted([c for c in os.listdir(split_path)
                          if os.path.isdir(os.path.join(split_path, c))])
        total_split = sum(
            len(os.listdir(os.path.join(split_path, c))) for c in classes
        )
        for classe in classes:
            nb = len(os.listdir(os.path.join(split_path, classe)))
            records.append({
                'dataset'       : dataset_name,
                'split'         : split,
                'classe'        : classe,
                'nb_images'     : nb,
                'total_split'   : total_split,
                'pourcentage'   : round(nb / total_split * 100, 2)
            })

    df = pd.DataFrame(records)
    path = os.path.join(OUTPUT_DIR, f"distribution_{dataset_name}.csv")
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"✅ Sauvegardé : {path}")
    print(df.to_string(index=False))
    return df

df_dist_oct  = export_distribution(OCT_PATH,  "OCT")
df_dist_xray = export_distribution(XRAY_PATH, "XRAY")

# CSV combiné OCT + XRAY
df_dist_all = pd.concat([df_dist_oct, df_dist_xray], ignore_index=True)
df_dist_all.to_csv(os.path.join(OUTPUT_DIR, "distribution_all.csv"),
                   index=False, encoding='utf-8-sig')
print("\n✅ CSV combiné sauvegardé : distribution_all.csv")

✅ Sauvegardé : ../powerbi\distribution_OCT.csv
dataset split classe  nb_images  total_split  pourcentage
    OCT train    CNV      32280       128393        25.14
    OCT train    DME      32280       128393        25.14
    OCT train DRUSEN      32280       128393        25.14
    OCT train NORMAL      31553       128393        24.58
    OCT   val    CNV       3798         9229        41.15
    OCT   val    DME       1335         9229        14.47
    OCT   val DRUSEN        941         9229        10.20
    OCT   val NORMAL       3155         9229        34.19
    OCT  test    CNV       6330        15373        41.18
    OCT  test    DME       2219        15373        14.43
    OCT  test DRUSEN       1568        15373        10.20
    OCT  test NORMAL       5256        15373        34.19
✅ Sauvegardé : ../powerbi\distribution_XRAY.csv
dataset split    classe  nb_images  total_split  pourcentage
   XRAY train    NORMAL       4329         8658        50.00
   XRAY train PNEUMONIA      

CSV 2 : Métadonnées image par image (taille, luminosité, contraste)

In [3]:
def export_metadata_images(base_path, dataset_name, max_par_classe=500):
    records = []

    for split in SPLITS:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            continue

        classes = sorted([c for c in os.listdir(split_path)
                          if os.path.isdir(os.path.join(split_path, c))])

        for classe in classes:
            classe_path = os.path.join(split_path, classe)
            fichiers    = sorted(os.listdir(classe_path))[:max_par_classe]

            for img_file in fichiers:
                img_path = os.path.join(classe_path, img_file)
                try:
                    pil_img  = Image.open(img_path).convert('RGB')
                    gray_img = np.array(pil_img.convert('L'))
                    w, h     = pil_img.size

                    # Stats canaux RGB
                    arr_rgb = np.array(pil_img)
                    r_mean  = round(arr_rgb[:,:,0].mean(), 2)
                    g_mean  = round(arr_rgb[:,:,1].mean(), 2)
                    b_mean  = round(arr_rgb[:,:,2].mean(), 2)

                    # Stats niveaux de gris
                    lum_mean = round(gray_img.mean(), 2)
                    lum_std  = round(gray_img.std(),  2)
                    contraste = int(gray_img.max()) - int(gray_img.min())

                    # Type image : originale ou augmentée
                    is_aug = 1 if img_file.startswith("aug_") else 0

                    records.append({
                        'dataset'         : dataset_name,
                        'split'           : split,
                        'classe'          : classe,
                        'fichier'         : img_file,
                        'largeur_px'      : w,
                        'hauteur_px'      : h,
                        'pixels_total'    : w * h,
                        'ratio_wh'        : round(w / h, 3),
                        'lum_moyenne'     : lum_mean,
                        'lum_std'         : lum_std,
                        'contraste'       : contraste,
                        'pixel_min'       : int(gray_img.min()),
                        'pixel_max'       : int(gray_img.max()),
                        'r_mean'          : r_mean,
                        'g_mean'          : g_mean,
                        'b_mean'          : b_mean,
                        'is_augmented'    : is_aug
                    })
                except Exception as e:
                    records.append({
                        'dataset': dataset_name, 'split': split,
                        'classe': classe, 'fichier': img_file,
                        'erreur': str(e)
                    })

    df = pd.DataFrame(records)
    path = os.path.join(OUTPUT_DIR, f"metadata_images_{dataset_name}.csv")
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"✅ {dataset_name} — {len(df)} lignes → {path}")
    print(df.describe().round(2))
    return df

df_meta_oct  = export_metadata_images(OCT_PATH,  "OCT",  max_par_classe=500)
df_meta_xray = export_metadata_images(XRAY_PATH, "XRAY", max_par_classe=500)

# CSV combiné
df_meta_all = pd.concat([df_meta_oct, df_meta_xray], ignore_index=True)
df_meta_all.to_csv(os.path.join(OUTPUT_DIR, "metadata_images_all.csv"),
                   index=False, encoding='utf-8-sig')
print("\n✅ CSV combiné sauvegardé : metadata_images_all.csv")

✅ OCT — 6000 lignes → ../powerbi\metadata_images_OCT.csv
       largeur_px  hauteur_px  pixels_total  ratio_wh  lum_moyenne  lum_std  \
count     6000.00     6000.00       6000.00   6000.00      6000.00  6000.00   
mean       657.24      499.79     327932.59      1.32        49.69    51.28   
std        284.19        6.81     140003.75      0.58        19.27    16.92   
min        512.00      496.00     253952.00      1.00        16.56    18.27   
25%        512.00      496.00     253952.00      1.03        36.65    38.30   
50%        512.00      496.00     262144.00      1.03        44.99    47.03   
75%        768.00      496.00     380928.00      1.55        56.60    60.46   
max       1536.00      512.00     761856.00      3.10       179.29   108.79   

       contraste  pixel_min  pixel_max   r_mean   g_mean   b_mean  \
count    6000.00     6000.0    6000.00  6000.00  6000.00  6000.00   
mean      255.00        0.0     255.00    49.69    49.69    49.69   
std         0.12        